In [49]:
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from unstructured.partition.pdf import partition_pdf
import psycopg2
from pgvector.psycopg2 import register_vector
from langchain_core.messages import SystemMessage, HumanMessage
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from ragas import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import PyPDFLoader

C:\Users\rayvi\AppData\Local\Temp\ipykernel_21108\4271915195.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\rayvi\AppData\Local\Temp\ipykernel_21108\4271915195.py:12: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\rayvi\AppData\Local\Temp\ipykernel_21108\4271915195.py:12: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

In [ ]:
test_df = pd.read_csv("ragas_testset.csv")
test_df

In [53]:
load_dotenv()

True

In [54]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [10]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini", rate_limiter=rate_limiter)

In [16]:
def chunk_by_structure():  
    chunks = partition_pdf(
            filename="pdfs/government-personal-data-protection-policies-jul21.pdf",
            strategy="hi_res", # table detection and extraction as text_as_html

            infer_table_structure=True,
            extract_image_block_types=["Image", "Table"],
            extract_image_block_to_payload=True, # extracts images and tables as base64 encoded strings

            chunking_strategy="by_title",
            max_characters=2000,
            combine_text_under_n_chars=1000,
            new_after_n_chars=1500
        )
    return [chunk.text for chunk in chunks]

In [50]:
def chunk_by_semantics():
    loader = PyPDFLoader("pdfs/government-personal-data-protection-policies-jul21.pdf")
    pages = loader.load()
    full_text = " ".join([p.page_content for p in pages])
    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90
    )
    chunks = text_splitter.create_documents([full_text])
    return chunks

In [55]:
chunk_text = chunk_by_semantics()

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2\nPlease retry in 56.350695271s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'q

docker exec -it pgvector-db psql -U postgres

In [20]:
conn = psycopg2.connect(
host="localhost",
port=5432,
database="rag_db",
user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [22]:
def storing(chunk_text):
    for text in chunk_text:
        vector = embeddings.embed_query(text)
        try:
            cur.execute('''
                INSERT INTO test_chunk_store (chunk_embedding, chunk_text)
                VALUES (%s, %s)
            ''', (vector, text))
            conn.commit()
        except Exception as e: 
            print(f"Postgres Error: {e}")
            conn.rollback()

In [24]:
storing(chunk_text)

In [27]:
def search_chunks(query):
    query_vector = embeddings.embed_query(query)
    search_query = '''
        SELECT chunk_text, 1 - (chunk_embedding <=> %s) AS similarity
        FROM test_chunk_store
        ORDER BY chunk_embedding <=> %s
        LIMIT 5;
    '''
    try:
        cur.execute(search_query, (str(query_vector), str(query_vector)))
        data = cur.fetchall()
        df = pd.DataFrame(data, columns=["chunk_text", "similarity"])
        return df
    except Exception as e:
        print(f"An error occurred: {e}")
        conn.rollback()
    

In [29]:
def filter(df, threshold=0.1):
    max_similarity = df.iloc[0]["similarity"]
    mask = df["similarity"] >= (max_similarity - threshold)

    final_df = df[mask]
    return final_df

In [30]:
def generate_context(dataframe):
    context_text = "\n\n".join(
        f"------\n{row['chunk_text']}"
        for _, row in dataframe.iterrows()
    )
    return context_text

In [32]:
def generate_message(query, context_text):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.
    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).
    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.
    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  
    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.
    6. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    HumanMessage(content=f"""
    Context: 
    {context_text}

    Question: {query}
    """),

    ]
    return messages

In [35]:
evaluation_data = []
for index, row in test_df.iterrows():
    question = row.user_input
    ground_truth = row.reference

    retrieved_df = search_chunks(question)
    filtered_df = filter(retrieved_df)
    context_text = filtered_df["chunk_text"].tolist()

    context = generate_context(retrieved_df)
    messages = generate_message(question, context)
    response = llm.invoke(messages)

    evaluation_data.append({
        "question": question,
        "answer": response.content,
        "contexts": context_text,
        "ground_truth": ground_truth
    })
eval_dataset = Dataset.from_list(evaluation_data)

In [36]:
df = eval_dataset.to_pandas()
df.head()

,question,answer,contexts,ground_truth
0,What governs the Personal Data Protection Poli...,The Personal Data Protection Policies are gove...,[This document sets out the key policies in th...,The Personal Data Protection Policies operate ...
1,Wat is the IM on ICT&SS Managment and how does...,### IM on ICT&SS Management\n\n- The IM on ICT...,[This document sets out the key policies in th...,The IM on ICT&SS Management is part of the Gov...
2,Wat is IM on ICT&SS Managment?,### IM on ICT&SS Management\n\n- The IM on ICT...,[This document sets out the key policies in th...,The IM on ICT&SS Management sets out how the G...
3,What is business contact information?,### Business Contact Information\n\nBusiness c...,[“business asset transaction” —\n\n(a) means t...,Business contact information is an individual’...
4,What rules agencies follow for Protection of P...,### Rules for Protection of Personal Data by A...,[This document sets out the key policies in th...,Agencies must ensure that up-to-date policies ...


In [44]:
config = RunConfig(
    timeout=180,
    max_retries=2
)
metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
]
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=LangchainEmbeddingsWrapper(embeddings),
    batch_size=2,
    run_config=config
)

C:\Users\rayvi\AppData\Local\Temp\ipykernel_21108\263520516.py:15: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings=LangchainEmbeddingsWrapper(embeddings),
Evaluating: 100%|██████████| 48/48 [09:56<00:00, 12.43s/it]


In [45]:
print(results)

{'faithfulness': 0.9009, 'answer_relevancy': 0.9234, 'context_recall': 0.8139, 'context_precision': 0.9088}


In [46]:
results_df = results.to_pandas()

In [47]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision
0,What governs the Personal Data Protection Poli...,[This document sets out the key policies in th...,The Personal Data Protection Policies are gove...,The Personal Data Protection Policies operate ...,0.666667,1.000000,1.000000,0.950000
1,Wat is the IM on ICT&SS Managment and how does...,[This document sets out the key policies in th...,### IM on ICT&SS Management\n\n- The IM on ICT...,The IM on ICT&SS Management is part of the Gov...,1.000000,0.965434,1.000000,1.000000
2,Wat is IM on ICT&SS Managment?,[This document sets out the key policies in th...,### IM on ICT&SS Management\n\n- The IM on ICT...,The IM on ICT&SS Management sets out how the G...,1.000000,0.818718,1.000000,1.000000
3,What is business contact information?,[“business asset transaction” —\n\n(a) means t...,### Business Contact Information\n\nBusiness c...,Business contact information is an individual’...,0.888889,0.956623,1.000000,0.366667
4,What rules agencies follow for Protection of P...,[This document sets out the key policies in th...,### Rules for Protection of Personal Data by A...,Agencies must ensure that up-to-date policies ...,0.800000,0.916090,0.500000,1.000000
5,What laws must an agency check before collecti...,"[Collection, Use and Disclosure of Personal Da...",### Laws an Agency Must Check Before Collectio...,"Before an agency collects personal data, it mu...",1.000000,0.984866,1.000000,1.000000
6,What are the conditions for personal data coll...,"[Collection, Use and Disclosure of Personal Da...",### Conditions for Personal Data Collection un...,"Under the Personal Data Protection Act (PDPA),...",0.944444,0.911662,0.666667,1.000000
7,How do the legal frameworks governing data man...,"[Collection, Use and Disclosure of Personal Da...",### Legal Frameworks for Data Management in th...,The legal frameworks governing data management...,0.636364,0.848000,0.600000,1.000000
8,What changes were made to the personal data pr...,[Government Personal\n\nData\n\nProtection Pol...,### Changes Made to Personal Data Protection P...,"In July 2021, the personal data protection pol...",0.875000,0.893048,1.000000,1.000000
9,What are the requirements for an agency to col...,"[Annex B - Additional Bases for Collection, Us...",### Requirements for Collecting Personal Data ...,An agency can collect personal data for resear...,1.000000,0.896944,0.500000,1.000000
